# Reusable Template — Leontief Input-Output Model

Swap in your own sectors, coefficients, and demand to analyze a **different**
economy (a country, a city, a supply chain) with the same NumPy techniques from
Chapter 3 of *Python for Engineering and Scientific Computing*.

**To reuse this notebook for a new project:**
1. Edit only the `# >>> EDIT HERE <<<` cell in Section 1 — everything below it
   is generic and runs unchanged.
2. Keep the number of sectors in `SECTOR_NAMES`, `A`, and `d` consistent
   (an *n*-sector economy needs an *n×n* matrix `A` and a length-*n* vector `d`).
3. Re-run the whole notebook (Kernel → Restart & Run All) after editing.

Read **Section 5 (Limitations)** before you trust or publish any output from
this template on a new dataset — the validity checks in Section 4 are necessary
but not sufficient for the model to be a good fit for your question.


In [ ]:
import numpy as np
from numpy.linalg import solve, inv, matrix_rank

np.set_printoptions(precision=3, suppress=True)


## 1. Inputs — edit this cell for your project

In [ ]:
# >>> EDIT HERE <<<
SECTOR_NAMES = ["Agriculture", "Manufacturing", "Services"]

# Technical coefficient matrix: A[i, j] = input from sector i needed
# to produce 1 unit of output in sector j
A = np.array([[0.10, 0.20, 0.05],
              [0.30, 0.10, 0.15],
              [0.15, 0.25, 0.10]])

# Final demand (consumers/exports/gov't), same units as A, one value per sector
d = np.array([100, 150, 120])

# Assumed demand uncertainty for the Monte Carlo step (same order as SECTOR_NAMES)
# Set to zeros if you don't want / don't have grounds to simulate uncertainty
DEMAND_STD = np.array([15, 25, 20])

N_SCENARIOS = 1000
RANDOM_SEED = 1
# >>> END EDIT <<<

n = len(SECTOR_NAMES)
assert A.shape == (n, n), "A must be square and match len(SECTOR_NAMES)"
assert d.shape == (n,), "d must have one entry per sector"
assert DEMAND_STD.shape == (n,), "DEMAND_STD must have one entry per sector"


## 2. Validity checks

These checks catch the most common reasons an input-output model is not usable
*at all* — they do not tell you whether the model is a good real-world fit
(see Section 5), only whether the math is solvable and internally consistent.

In [ ]:
I_n = np.eye(n)
system_matrix = I_n - A

# Check 1: coefficients should be non-negative
assert np.all(A >= 0), "Technical coefficients must be non-negative."

# Check 2: each column sum should be < 1 (a sector cannot need more than
# $1 of total inputs to make $1 of output, or the economy is not viable)
col_sums = A.sum(axis=0)
if np.any(col_sums >= 1):
    print("WARNING: column sum >= 1 for sector(s):",
          [SECTOR_NAMES[i] for i in np.where(col_sums >= 1)[0]])
    print("This economy cannot produce a viable surplus with these coefficients.")

# Check 3: (I - A) must be invertible (non-singular)
rank = matrix_rank(system_matrix)
if rank < n:
    raise ValueError("(I - A) is singular — cannot solve. Check for duplicate "
                      "or degenerate sector definitions in A.")

print("Validity checks passed. Column sums of A:", np.round(col_sums, 3))


## 3. Solve the static model

In [ ]:
x = solve(system_matrix, d)

print("Required total output per sector:")
for s, val in zip(SECTOR_NAMES, x):
    print(f"  {s:16s}: {val:10.2f}")

Z = A @ np.diag(x)  # inter-industry dollar flows
print("\nInter-industry transaction matrix Z:\n", np.round(Z, 2))


## 4. Monte Carlo sensitivity to demand uncertainty

Only meaningful if `DEMAND_STD` reflects a real, defensible estimate of demand
uncertainty (see Section 5) rather than an arbitrary guess.

In [ ]:
def simulate_output(A, d, demand_std, n_scenarios=1000, seed=None):
    """Simulate `n_scenarios` demand draws (independent normal per sector)
    and return the resulting sector outputs for every scenario.

    Returns
    -------
    d_sim : ndarray, shape (n_scenarios, n_sectors)
    x_sim : ndarray, shape (n_scenarios, n_sectors)
    """
    if seed is not None:
        np.random.seed(seed)
    n_sectors = len(d)
    d_sim = np.column_stack([
        np.random.normal(d[i], demand_std[i], n_scenarios)
        for i in range(n_sectors)
    ])
    inv_system = inv(np.eye(n_sectors) - A)
    x_sim = d_sim @ inv_system.T
    return d_sim, x_sim


d_sim, x_sim = simulate_output(A, d, DEMAND_STD, N_SCENARIOS, RANDOM_SEED)

print(f"=== Monte Carlo results ({N_SCENARIOS} scenarios) ===")
for i, s in enumerate(SECTOR_NAMES):
    col = x_sim[:, i]
    print(f"{s:16s} mean={np.mean(col):9.2f}  std={np.std(col):7.2f}  "
          f"min={np.amin(col):9.2f}  max={np.amax(col):9.2f}")

total_output = np.sum(x_sim, axis=1)
best_idx = np.where(total_output == np.amax(total_output))[0][0]
print(f"\nScenario with highest total output: #{best_idx} "
      f"(total = {total_output[best_idx]:.2f})")

cv = np.std(x_sim, axis=0) / np.mean(x_sim, axis=0)
print("Coefficients of variation:", np.round(cv, 3))
print("Most volatile sector:", SECTOR_NAMES[np.argmax(cv)])

# Flag any simulated scenario with a negative output — a sign the model is
# being pushed outside a realistic demand range
neg_scenarios = np.where(np.any(x_sim < 0, axis=1))[0]
print(f"\nScenarios with a negative sector output: {len(neg_scenarios)} "
      f"of {N_SCENARIOS} ({100*len(neg_scenarios)/N_SCENARIOS:.1f}%)")


## 5. Limitations — read before reusing on a new dataset

This template is only ever a **linear, static, fixed-technology** model. That
single sentence is the source of every limitation below.

### ✅ Reasonable to use when
- You need a **first-pass, short-run** estimate of how a change in demand
  ripples through a small number of well-defined, interdependent sectors.
- The technical coefficients in `A` are based on real, recent input-output
  data for the economy you're modeling (e.g., published national/regional
  input-output tables), not guesses.
- You're comparing **relative** outcomes across scenarios (which sector grows
  most, which is most sensitive) rather than claiming precise absolute
  dollar forecasts.
- The time horizon is short enough that production technology and trade
  patterns are unlikely to change materially.
- You clearly label Monte Carlo results as *what the model produces under the
  assumed uncertainty*, not as a validated probabilistic forecast.

### 🚫 Not recommended when
- **Long-run forecasting.** `A` is assumed fixed; real economies change
  technology, automate, and substitute inputs over years/decades.
- **Structural breaks are present or expected** (recession, pandemic, war,
  major policy shift, new technology). Coefficients estimated from
  "before" data will not describe "after."
- **Prices and behavior matter to your question.** This model has no prices,
  no elasticities, no substitution — sectors cannot respond to becoming more
  expensive or scarce. Use a Computable General Equilibrium (CGE) model or
  econometric model instead if that's the question.
- **You don't have real data for `A`.** Coefficients invented for a demo (as
  in this template) are fine for *learning the technique* — they are not
  fine for a real policy or investment decision.
- **The Monte Carlo standard deviations (`DEMAND_STD`) are guesses, not
  estimated from data.** Garbage-in-garbage-out applies directly: the
  simulated "uncertainty" only reflects the width you typed in.
- **Extreme/tail events matter to your question.** The normal distribution
  used for demand has thin tails and will systematically underestimate the
  probability of large shocks (e.g., a pandemic-scale demand collapse).
- **Any simulated output goes negative or a validity check in Section 2
  failed/warned.** That's the model telling you the demand range you fed
  it, or the coefficients themselves, are outside where the linear
  approximation still makes sense.
- **You need causal inference** ("did policy X cause sector Y to grow?").
  This model is descriptive/mechanical, not causal — it cannot separate
  correlation from causation and has no error term to test.
